In [6]:
# 01_preprocessing.ipynb
# =============================================================================
# Personalized Health Intervention Pathways for Comorbid Policyholders:
# Loss Ratio Management via LLM Guardrails and Counterfactual Explanations
# -----------------------------------------------------------------------------
# Notebook 01 — Data Preprocessing & Feature Engineering
# Data   : KNHANES (Korea National Health and Nutrition Examination Survey)
#          2020–2024 (5 cycles merged)
# Output : knhanes_comorbid_2020_2024.csv  — analysis-ready dataset
# =============================================================================

# %%
# ## 0. Import Libraries

import numpy as np
import pandas as pd
import pyreadstat

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)
pd.set_option('display.max_colwidth', None)

# %%
# ## 1. Age-Group × Sex Stratification Configuration
# Four stratified groups:
#   MiddleAged (40–59) × Male / Female
#   Older      (60+)   × Male / Female

AGEGROUP_CONFIG = {
    "MiddleAged_Male":   {"age_min": 40, "age_max": 59, "sex_code": 1.0},
    "MiddleAged_Female": {"age_min": 40, "age_max": 59, "sex_code": 2.0},
    "Older_Male":        {"age_min": 60, "age_max": 99, "sex_code": 1.0},
    "Older_Female":      {"age_min": 60, "age_max": 99, "sex_code": 2.0},
}

# %%
# ## 2. Load and Merge KNHANES 2020–2024

DATA_DIR = '../data/'

df20, _ = pyreadstat.read_sas7bdat(DATA_DIR + 'hn20_all.sas7bdat')
df21, _ = pyreadstat.read_sas7bdat(DATA_DIR + 'hn21_all.sas7bdat')
df22, _ = pyreadstat.read_sas7bdat(DATA_DIR + 'hn22_all.sas7bdat')
df23, _ = pyreadstat.read_sas7bdat(DATA_DIR + 'hn23_all.sas7bdat')
df24, _ = pyreadstat.read_sas7bdat(DATA_DIR + 'hn24_all.sas7bdat')

# %%
# ## 3. Variable Selection

KEY_COLS    = ['ID', 'year', 'sex', 'age']
CAT_COLS    = [
    'HE_obe', 'BO1_1', 'BO1_2', 'BO1_3', 'BD1_11', 'BD2_1', 'BS3_1',
    'BE3_71', 'BE3_75', 'BE3_81', 'BE3_91', 'pa_aerobic', 'L_BR_FQ',
    'BP1', 'mh_stress', 'incm', 'ho_incm', 'edu', 'BH1',
]
NUM_COLS    = [
    'HE_BMI', 'HE_wc', 'HE_wt',
    'N_EN', 'N_CHO', 'N_SUGAR', 'N_NA', 'N_FAT',
    'N_SFA', 'N_TDF', 'N_K', 'N_PROT',
]
TARGET_COLS = ['HE_DM_HbA1c', 'HE_HP']
ALL_VARS    = KEY_COLS + CAT_COLS + NUM_COLS + TARGET_COLS

# %%
# ## 4. Filter Columns per Year and Merge

df_total = pd.concat(
    [d[[v for v in ALL_VARS if v in d.columns]].copy()
     for d in [df20, df21, df22, df23, df24]],
    axis=0
).reset_index(drop=True)

print(f"Merged dataset shape: {df_total.shape}")

# %%
# ## 5. Binary Encoding of Target Variables
# HE_DM_HbA1c : 1 = Normal → 0 | 3 = Diabetes → 1
# HE_HP        : 1 = Normal → 0 | 4 = Hypertension → 1

df_total['HE_DM_HbA1c'] = df_total['HE_DM_HbA1c'].map({1: 0, 3: 1}).fillna(-999)
df_total['HE_HP']        = df_total['HE_HP'].map({1: 0, 4: 1}).fillna(-999)

for col in ['HE_DM_HbA1c', 'HE_HP']:
    print(f"\n{col} distribution:\n", df_total[col].value_counts())

# %%
# ## 6. Risk Class Label Generation
# Class 0: Normal       — neither condition
# Class 1: HTN-only     — hypertension only
# Class 2: DM-only      — diabetes only
# Class 3: Comorbid     — both hypertension and diabetes (primary intervention target)

conditions = [
    (df_total['HE_DM_HbA1c'] == 0) & (df_total['HE_HP'] == 0),
    (df_total['HE_DM_HbA1c'] == 0) & (df_total['HE_HP'] == 1),
    (df_total['HE_DM_HbA1c'] == 1) & (df_total['HE_HP'] == 0),
    (df_total['HE_DM_HbA1c'] == 1) & (df_total['HE_HP'] == 1),
]
df_total['integrated_target'] = np.select(conditions, [0, 1, 2, 3], default=np.nan)
df_total = df_total.dropna().reset_index(drop=True)
df_total['integrated_target'] = df_total['integrated_target'].astype(int)

print("\nRisk class distribution (full sample):")
print(df_total['integrated_target'].value_counts().sort_index())

# %%
# ## 7. Age Filtering and Age-Group Variable
# Restrict to age >= 40 (underwriting target population)
# AgeGroup: 0 = MiddleAged (40–59) | 1 = Older (60+)

if 'age' not in df_total.columns:
    raise ValueError(
        "'age' column not found. Check KNHANES variable name (e.g. age, HE_age)."
    )

df_total['age'] = pd.to_numeric(df_total['age'], errors='coerce')
df_total = df_total[df_total['age'] >= 40].copy().reset_index(drop=True)
df_total['age_group'] = np.where(df_total['age'] < 60, 0, 1)

print("\nAge-group distribution (0 = MiddleAged 40–59, 1 = Older 60+):")
print(df_total['age_group'].value_counts().sort_index())

# %%
# ## 8. Rename Columns: SAS Raw Codes → English Labels

RENAME_DICT = {
    'ID':                'ID',
    'year':              'SurveyYear',
    'sex':               'Sex',
    'age':               'Age',
    'age_group':         'AgeGroup',
    'HE_DM_HbA1c':       'Diabetes',
    'HE_HP':             'Hypertension',
    'integrated_target': 'RiskClass',
    # Categorical
    'HE_obe':    'ObesityStatus',
    'BO1_1':     'WeightChangeStatus',
    'BO1_2':     'WeightLossAmount',
    'BO1_3':     'WeightGainAmount',
    'BD1_11':    'DrinkingFrequency',
    'BD2_1':     'DrinkingAmount',
    'BS3_1':     'SmokingStatus',
    'BE3_71':    'VigorousActivity_Work',
    'BE3_75':    'VigorousActivity_Leisure',
    'BE3_81':    'ModerateActivity_Work',
    'BE3_91':    'WalkingActivity',
    'pa_aerobic':'AerobicActivityRate',
    'L_BR_FQ':   'BreakfastFrequency',
    'BP1':       'StressLevel',
    'mh_stress': 'StressAwarenessRate',
    'incm':      'PersonalIncomeQuartile',
    'ho_incm':   'HouseholdIncomeQuartile',
    'edu':       'EducationLevel',
    'BH1':       'HealthScreeningStatus',
    # Numerical
    'HE_BMI':  'BMI',
    'HE_wc':   'WaistCirc',
    'HE_wt':   'Weight',
    'N_EN':    'Energy_kcal',
    'N_CHO':   'Carb_g',
    'N_SUGAR': 'Sugar_g',
    'N_NA':    'Sodium_mg',
    'N_FAT':   'Fat_g',
    'N_SFA':   'SaturatedFat_g',
    'N_TDF':   'Fiber_g',
    'N_K':     'Potassium_mg',
    'N_PROT':  'Protein_g',
}

df_total.rename(columns=RENAME_DICT, inplace=True)

# %%
# ## 9. Risk Class Distribution by Age-Group × Sex
# Primary reference table for manuscript Table 1

print("\nRisk class distribution by AgeGroup × Sex:")
print("(AgeGroup: 0 = MiddleAged, 1 = Older | Sex: 1 = Male, 2 = Female)\n")
dist_table = (
    df_total
    .groupby(['AgeGroup', 'Sex', 'RiskClass'])
    .size()
    .unstack(fill_value=0)
)
print(dist_table)

# %%
# ## 10. Save Analysis-Ready Dataset

df_total.to_csv('../results/tables/knhanes_comorbid_2020_2024.csv', index=False, encoding='utf-8-sig')
print("\n>>> Preprocessing complete.")
print(f"    Final dataset shape : {df_total.shape}")
print(f"    Saved to            : ../results/tables/knhanes_comorbid_2020_2024.csv")
df_total.head()

Merged dataset shape: (34640, 37)

HE_DM_HbA1c distribution:
 HE_DM_HbA1c
-999.0    17438
 0.0      12810
 1.0       4392
Name: count, dtype: int64

HE_HP distribution:
 HE_HP
-999.0    16734
 0.0      11964
 1.0       5942
Name: count, dtype: int64

Risk class distribution (full sample):
integrated_target
0    6381
1    1370
2     649
3    1338
Name: count, dtype: int64

Age-group distribution (0 = MiddleAged 40–59, 1 = Older 60+):
age_group
0    3360
1    2979
Name: count, dtype: int64

Risk class distribution by AgeGroup × Sex:
(AgeGroup: 0 = MiddleAged, 1 = Older | Sex: 1 = Male, 2 = Female)

RiskClass        0    1    2    3
AgeGroup Sex                     
0        1.0   615  196  106  171
         2.0  1785  242  143  102
1        1.0   260  344  194  526
         2.0   465  505  170  515

>>> Preprocessing complete.
    Final dataset shape : (6339, 39)
    Saved to            : ../results/tables/knhanes_comorbid_2020_2024.csv


,ID,SurveyYear,Sex,Age,ObesityStatus,WeightChangeStatus,WeightLossAmount,WeightGainAmount,DrinkingFrequency,DrinkingAmount,SmokingStatus,VigorousActivity_Work,VigorousActivity_Leisure,ModerateActivity_Work,WalkingActivity,AerobicActivityRate,BreakfastFrequency,StressLevel,StressAwarenessRate,PersonalIncomeQuartile,HouseholdIncomeQuartile,EducationLevel,HealthScreeningStatus,BMI,WaistCirc,Weight,Energy_kcal,Carb_g,Sugar_g,Sodium_mg,Fat_g,SaturatedFat_g,Fiber_g,Potassium_mg,Protein_g,Diabetes,Hypertension,RiskClass,AgeGroup
0,A801247601,2020.0,1.0,70.0,2.0,1.0,8.0,8.0,5.0,3.0,3.0,2.0,2.0,2.0,2.0,0.0,1.0,3.0,0.0,3.0,2.0,4.0,1.0,19.090342,75.6,57.4,2173.580968,343.730353,132.566925,4824.388890,63.119559,22.552808,27.774115,3122.056504,68.289139,0.0,0.0,0,1
1,A801364901,2020.0,2.0,57.0,2.0,1.0,8.0,8.0,1.0,8.0,8.0,2.0,2.0,2.0,1.0,1.0,3.0,4.0,0.0,3.0,4.0,3.0,1.0,19.846405,73.6,50.3,1356.052087,252.113469,38.811711,2253.135696,16.232627,4.449380,22.044980,1808.123847,45.851369,1.0,0.0,2,0
2,A802276501,2020.0,1.0,58.0,3.0,1.0,8.0,8.0,3.0,1.0,8.0,2.0,2.0,2.0,1.0,1.0,1.0,3.0,0.0,3.0,4.0,4.0,2.0,24.492911,86.8,68.8,2321.054277,466.029105,163.377779,4575.544955,31.941538,7.975558,53.294924,4577.031940,59.289975,0.0,0.0,0,0
3,A802276502,2020.0,2.0,57.0,2.0,1.0,8.0,8.0,8.0,8.0,8.0,2.0,2.0,2.0,1.0,0.0,1.0,1.0,1.0,4.0,4.0,4.0,1.0,21.101375,75.5,50.5,1103.432124,186.759060,74.954384,2423.838154,19.885312,5.949561,26.034008,2929.959228,49.783588,0.0,0.0,0,0
4,A802334301,2020.0,2.0,40.0,2.0,1.0,8.0,8.0,5.0,1.0,8.0,2.0,2.0,2.0,1.0,1.0,1.0,3.0,0.0,4.0,4.0,4.0,1.0,18.623267,63.2,47.2,1675.946591,305.140783,68.298590,3465.478952,27.455917,12.184819,29.264159,3548.158736,49.850564,0.0,0.0,0,0
